# Phase 3: Reconstruct a Synthetic User ID 

In [5]:
import pandas as pd
import numpy as np

DATA_IN  = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_merged.parquet"
DATA_OUT = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_with_uid.parquet"

## 1. Load

In [6]:
df = pd.read_parquet(DATA_IN)
print(f"Shape: {df.shape}")

Shape: (590540, 434)


## 2. Build uid

card_age_day = floor(TransactionDT / 86400) - D1
uid          = card1 | addr1 | P_emaildomain | card_age_day

In [ ]:
# Approximate card issue day
df["card_age_day"] = (df["TransactionDT"] / 86400).astype(int) - df["D1"].fillna(-1).astype(int)

# Build uid as a string concat of stable entity fields
def to_str(col):
    return df[col].astype("string").fillna("NA")

df["uid"] = (
    to_str("card1") + "_" +
    to_str("addr1") + "_" +
    to_str("P_emaildomain") + "_" +
    df["card_age_day"].astype("string")
)

print(f"Unique uids: {df['uid'].nunique():,}")
print(f"Mean transactions per uid: {len(df) / df['uid'].nunique():.2f}")

Unique uids: 273,825
Mean transactions per uid: 2.16


## 3. Check uid quality

In [ ]:
g = df.groupby("uid")["isFraud"].agg(["count", "mean"])

single_txn_uids = (g["count"] == 1).mean()
multi_txn = g[g["count"] >= 2]
pure_uids = ((multi_txn["mean"] == 0) | (multi_txn["mean"] == 1)).mean()

print(f"Share of uids w/ only 1 transaction : {single_txn_uids:.1%}")
g.describe()

Share of uids with only 1 transaction : 64.8%
Among multi-txn uids, share label-pure : 97.7%
(Label purity should be high — e.g. >95%. If not, try adding card2 or D3 to the uid.)


,count,mean
count,273825.000000,273825.000000
mean,2.156633,0.029491
std,4.234597,0.164491
min,1.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,2.000000,0.000000
max,1414.000000,1.000000


## 4. Save

In [ ]:
df.to_parquet(DATA_OUT, index=False)
print(f"Saved {DATA_OUT}")

Saved /Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_with_uid.parquet
Next step: 04_graph_features.ipynb
